# TN3 — Quét toàn dải alpha, DS-TCN 64 kênh tầm nhìn 121

Cấu hình thứ ba của TN3, bên cạnh `c64 k3 RF61` và `c192 k5 RF121`.

## Vì sao chạy thêm cấu hình này

Hai cấu hình TN3 đã xong cho cùng một kết luận về phần "có lai thì hơn MSE
thuần" — **20/20 mức alpha**, ở hai kích cỡ model cách nhau 8,3 lần tham số.
Nhưng chúng **không** cho cùng thứ hạng: tốp 3 của c64 k3 là 0,6 · 0,5 · 0,4
còn của c192 k5 là 0,2 · 0,3 · 0,0, không giao nhau mức nào.

Cấu hình này tách được hai biến đang dính vào nhau. So với `c64 k3` nó khác
đúng **tầm nhìn**; so với `c192 k5` nó khác đúng **số kênh**. Nên bảng thứ ba
trả lời được: thứ hạng alpha đổi theo tầm nhìn, theo số kênh, hay chỉ là nhiễu.

## Cấu hình nền

| | |
|---|---|
| model | `ds_tcn --channels 64 --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element` |
| tham số | **38.105** |
| tầm nhìn | 121 — phủ 60% cửa sổ vào, và 121% một pha thở |
| MSE thuần, alpha 1,0 | **0,757855** *(TN2, 4 fold, 1 seed)* |

`alpha = 1,0` **không chạy** — đó chính là MSE thuần, TN2 đã đo bằng đúng cấu
hình và đúng bốn fold này. Ô so sánh ở mục 5 tự chèn nó làm điểm tham chiếu.

## Chi phí

Đo từ dấu thời gian của TN3 c64 đã chạy: **khoảng 15 phút một fold**, tức
**~1 giờ một mức alpha**. Mười mức khoảng **10 giờ**.

**Không cần chạy hết một lần.** Mỗi alpha một ô riêng, và ô train nào cũng nén
lên Drive ngay sau khi xong. Dừng lúc nào cũng được: mở lại, chạy ô khôi phục ở
mục 1 rồi bấm tiếp ô còn thiếu. `run_cv.py` bỏ qua mọi fold đã xong.

Muốn thấy hình dạng sớm thì chạy thưa trước: **0,0 → 0,5 → 0,9 → 0,2 → 0,7**,
rồi lấp các điểm còn lại sau.

## Đọc kết quả

Một seed. `seed_std` của các cấu hình TN1 trải 0,0007–0,0108, mà biên độ dao
động bên trong mỗi cột TN3 đã đo được là 0,0151 và 0,0107 — cùng cỡ. Nên
**chênh lệch giữa hai alpha liền kề không đọc được**; thứ đọc được là *mọi mức
alpha có hơn MSE thuần hay không*, và *hình dạng chung của đường cong*.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

Khôi phục mọi mức alpha đã chạy.

**Chạy ô này mỗi khi mở lại notebook.** Nó gộp `runs/*/summary.csv` vào `runs/summary.csv` — chỗ `run_cv.py` tra để biết fold nào đã xong. Mẫu tên tệp bắt cả `c64_k3` lẫn `c64_k5`; hai bộ có `run_id` khác nhau nên gộp chung vô hại, còn bảng ở mục 5 lọc riêng `_c64_k5_`.

In [ ]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn3_*c64*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm bản cài đặt

Số tham số phải ra đúng **38.105**.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Mười mức alpha, mỗi mức đủ 4 fold

Mỗi ô train xong tự nén lên Drive, nên đứt phiên chỉ mất mức đang chạy.

**alpha 0,0 — MSE 0 phần trăm, Pearson 100 phần trăm**  — **Pearson thuần**, khớp hoàn hảo với cách chấm điểm

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.0 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,1 — MSE 10 phần trăm, Pearson 90 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.1 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,2 — MSE 20 phần trăm, Pearson 80 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.2 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,3 — MSE 30 phần trăm, Pearson 70 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.3 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,4 — MSE 40 phần trăm, Pearson 60 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.4 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,5 — MSE 50 phần trăm, Pearson 50 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.5 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,6 — MSE 60 phần trăm, Pearson 40 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.6 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,7 — MSE 70 phần trăm, Pearson 30 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.7 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,8 — MSE 80 phần trăm, Pearson 20 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.8 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

**alpha 0,9 — MSE 90 phần trăm, Pearson 10 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.9 --seed 0
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c64_k5

## 4. Bảng alpha, kèm điểm tham chiếu

Cột bên phải là chênh so với MSE thuần của chính cấu hình này.

In [ ]:
import csv, re, os
MSE_THUAN = 0.757855          # TN2, cùng cấu hình, 4 fold, 1 seed
d = {}
if os.path.exists("runs/tn3/summary.csv"):
    d = {float(re.search(r"_a([\d.]+)_", r["run_id"]).group(1)): float(r["score_macro"])
         for r in csv.DictReader(open("runs/tn3/summary.csv"))
         if r["fold"] == "TONG" and "_c64_k5_" in r["run_id"]}
d[1.0] = MSE_THUAN
for a in sorted(d):
    hon = "" if a == 1.0 else "  %+.4f" % (d[a] - MSE_THUAN)
    print("  alpha", a, " ", round(d[a], 6), "<- MSE thuần" if a == 1.0 else hon)

## 5. Đường cong alpha

In [ ]:
import matplotlib.pyplot as plt
x = sorted(d)
plt.plot(x, [d[i] for i in x], "o-", label="quét alpha")
plt.plot([1.0], [MSE_THUAN], "ro", ms=10, label="MSE thuần (đã có)")
plt.xlabel("alpha — trọng số MSE"); plt.ylabel("cv_score")
plt.legend(); plt.grid(alpha=.3); plt.title("0 = Pearson thuần, 1 = MSE thuần")

## 6. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()